# 03 - SIAT-LLMD Preprocessing

This notebook loads SIAT-LLMD CSVs and prepares knee angle/torque data.
Update the column names below to match your files.


In [1]:
import sys
from pathlib import Path
import yaml

ROOT = Path(r"e:/Optimal_Control/PINN/hjb_pinn_exoskeleton")
sys.path.append(str(ROOT))

from src.datasets import load_siat_dataset, find_csv_files

cfg = yaml.safe_load((ROOT / "configs" / "knee_config.yaml").read_text())
base = cfg["data_paths"]["siat_llmd_base"]

# Select side: "right" or "left"
SIDE = "right"
KIN_COL = f"Kinematic: {SIDE} knee flexion angle"
KINETIC_COL = f"Kinetic: {SIDE} knee flexion torque"

print("Base path:", base)


Base path: E:/Dissertation/EMG_DL_Nour/SIAT_LLMD20230404/SIAT_LLMD20230404


In [2]:
# Preview columns from the first WAK file
try:
    files = find_csv_files(base, "WAK")
    if not files:
        print("No WAK files found.")
    else:
        import pandas as pd
        print("Sample file:", files[0])
        df = pd.read_csv(files[0])
        print("Columns:")
        for c in df.columns:
            print(" -", c)
except Exception as e:
    print("Preview failed:", e)


Sample file: E:\Dissertation\EMG_DL_Nour\SIAT_LLMD20230404\SIAT_LLMD20230404\Sub01\Data\Sub01_WAK_Data.csv
Columns:
 - Time
 - Kinematic: left hip adduction angle
 - Kinematic: left hip flexion angle
 - Kinematic: left knee flexion angle
 - Kinematic: left ankle flexion angle
 - Kinematic: right hip adduction angle
 - Kinematic: right hip flexion angle
 - Kinematic: right knee flexion angle
 - Kinematic: right ankle flexion angle
 - Kinetic: left hip adduction torque
 - Kinetic: left hip flexion torque
 - Kinetic: left knee flexion torque
 - Kinetic: left ankle flexion torque
 - Kinetic: right hip adduction torque
 - Kinetic: right hip flexion torque
 - Kinetic: right knee flexion torque
 - Kinetic: right ankle flexion torque
 - sEMG: tensor fascia lata
 - sEMG: rectus femoris
 - sEMG: vastus medialis
 - sEMG: semimembranosus
 - sEMG: upper tibialis anterior
 - sEMG: lower tibialis anterior
 - sEMG: lateral gastrocnemius
 - sEMG: medial gastrocnemius
 - sEMG: soleus


In [3]:
try:
    samples = load_siat_dataset(base, KIN_COL, KINETIC_COL, file_filter="WAK_Data")
    print(f"Loaded {len(samples)} files")
except Exception as e:
    print("SIAT loading failed:", e)

Loaded 40 files


In [4]:
from scipy.signal import find_peaks
from src.datasets import SiatSample
import numpy as np

# Gait-cycle segmentation by knee angle peaks
# Settings
N_POINTS = 101  # resample each cycle to fixed length
MIN_PEAK_DISTANCE = None  # set to int if needed
PROMINENCE_FRAC = 0.1  # fraction of signal range

siat_cycles = []

for s in samples:
    theta = s.theta
    theta = np.deg2rad(theta)
    torque = s.torque

    if len(theta) < 10:
        continue

    prominence = PROMINENCE_FRAC * (theta.max() - theta.min() + 1e-6)
    distance = MIN_PEAK_DISTANCE
    if distance is None:
        distance = max(10, len(theta) // 50)

    peaks, _ = find_peaks(theta, distance=distance, prominence=prominence)

    # Need at least two peaks to define cycles
    if len(peaks) < 2:
        continue

    for i in range(len(peaks) - 1):
        i0, i1 = peaks[i], peaks[i + 1]
        if i1 - i0 < 5:
            continue

        seg_theta = theta[i0:i1]
        seg_torque = torque[i0:i1]

        # Normalize time to [0,1] and resample to fixed length
        t_old = np.linspace(0.0, 1.0, len(seg_theta))
        t_new = np.linspace(0.0, 1.0, N_POINTS)

        theta_rs = np.interp(t_new, t_old, seg_theta)
        torque_rs = np.interp(t_new, t_old, seg_torque)

        # Estimate omega from resampled theta (dt = 1/(N_POINTS-1))
        dt = 1.0 / (N_POINTS - 1)
        omega_rs = np.gradient(theta_rs, dt)

        siat_cycles.append(
            SiatSample(theta=theta_rs.astype(np.float32),
                       omega=omega_rs.astype(np.float32),
                       torque=torque_rs.astype(np.float32),
                       time=t_new.astype(np.float32))
        )

print(f"Segmented {len(siat_cycles)} gait cycles")

# Overwrite samples with segmented cycles for saving
samples = siat_cycles


Segmented 657 gait cycles


In [5]:
import pickle

if 'samples' in globals() and samples:
    out_path = ROOT / 'data' / 'siat_clean.pkl'
    with out_path.open('wb') as f:
        pickle.dump(samples, f)
    print(f"Saved: {out_path}")
else:
    print("No samples to save.")


Saved: e:\Optimal_Control\PINN\hjb_pinn_exoskeleton\data\siat_clean.pkl


## Next Steps
- Add gait-cycle segmentation (heel strike detection or % gait).
- Normalize time to [0, 1] per cycle.
- Save cleaned data to `data/siat_clean.pkl`.
